In [45]:
import requests
import torch
import re
import time
import psutil
import subprocess
from unidecode import unidecode
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, hamming_loss

import pandas as pd

from datasets import load_dataset

In [46]:
ds = load_dataset("higopires/RePro-categories-multilabel")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1007 entries, 0 to 1006
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   review_text             1007 non-null   object
 1   ENTREGA                 1007 non-null   int64 
 2   OUTROS                  1007 non-null   int64 
 3   PRODUTO                 1007 non-null   int64 
 4   CONDICOESDERECEBIMENTO  1007 non-null   int64 
 5   INADEQUADA              1007 non-null   int64 
 6   ANUNCIO                 1007 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 55.2+ KB


In [47]:
test = test[test['INADEQUADA'] == 0].reset_index(drop=True)

test = test.drop(columns=['INADEQUADA'])

test.rename(columns={'review_text': 'text'}, inplace=True)

test

,text,ENTREGA,OUTROS,PRODUTO,CONDICOESDERECEBIMENTO,ANUNCIO
0,"ESSE PRODUTO PODE ATÉ SER BOM, PORÉM, A AMERIC...",1,1,0,0,0
1,Recomendo!Os aparelhos da motorola são muito b...,0,0,1,0,0
2,"Eu ameiii o produto, pena que veio com o espe...",0,0,1,1,0
3,bom..............................................,0,0,1,0,0
4,Quero saber quando chegará mais pois gostaria ...,0,1,0,0,0
...,...,...,...,...,...,...
961,Produto veio com peças totalmente inferiores a...,0,0,0,1,1
962,Recebi Produto diferente do anunciado - com ap...,0,0,0,1,1
963,No anúncio constava três refis. Foi entregue a...,0,0,0,1,1
964,Comprei o produto na cor que está publicado no...,0,0,0,1,1


In [48]:
test.rename(columns={'CONDICOESDERECEBIMENTO': 'CONDICOES DE RECEBIMENTO'}, inplace=True)

labels = test.columns[1:]

labels

Index(['ENTREGA', 'OUTROS', 'PRODUTO', 'CONDICOES DE RECEBIMENTO', 'ANUNCIO'], dtype='object')

In [49]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_13040\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


97546240

In [50]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [52]:
def classify(text, labels):
    url = "http://localhost:11434/api/chat"
    
    messages = [
        {"role": "system", "content": "Você é um assistente de classificação. Seu objetivo é ler o texto fornecido e classificá-lo de acordo com a tarefa e os rótulos descritos. Você é capaz de lidar com tarefas de classificação multilabel com base nas instruções do user."},
        {"role": "user", "content": f"Classifique o seguinte texto com base na tarefa: Análise de categorias de avaliações de produtos e-commerce. Responda apenas com os rótulos que melhor descrevem o texto. Se houver mais de um rótulo, os separe com vírgula. Os rótulos possíveis são: {', '.join(labels)}. Texto: {text}"}
    ]
    
    start_time = time.time()

    try:
        response = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": True,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 3100
            }
        }, timeout=30)
        response_time = time.time() - start_time
        vram_usage = get_gpu_memory_usage()
        ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)
        response = response.json()
        response_text = response['message'].get('thinking', '') if 'message' in response else ''
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return "error", {}, 0, 0, 0, 0, f"API Error: {e}"
    
    if not response.get('done', False):
        print(f"Ollama returned an incomplete response: {response.get('error')}")
        return 'error', {}, response_time, 0, 0, 0, response.get('error', 'Incomplete response')
    
    if 'message' in response and 'content' in response['message']:
        classification_text = response['message']['content'].lower()
        print("Response fields:", ', '.join(response.keys()))
        print(response)
        total_time = response['total_duration'] / 1_000_000_000
    else:
        messages.append({"role": "assistant", "content": response_text + '</think>'})
        start_time2 = time.time()
        response2 = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": False,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 430
            }
        })
        response_time += time.time() - start_time2
        response2 = response2.json()
        classification_text = response2['message']['content'].lower() if 'message' in response2 and 'content' in response2['message'] else ''
        total_time = response['total_duration'] / 1_000_000_000 + response2['total_duration'] / 1_000_000_000
    
    label_counts = {label: len(re.findall(r'\b' + re.escape(label.lower()) + r'\b', classification_text)) for label in labels}

    list = []

    if 'entrega' in classification_text:
        list.append('ENTREGA')
    if 'produtos' in classification_text:
        list.append('PRODUTOS')
    if 'condicoes de recebimento' in classification_text:
        list.append('CONDICOES DE RECEBIMENTO')
    if 'anuncio' in classification_text:
        list.append('ANUNCIO')
    if 'outros' in classification_text:
        list.append('OUTROS')
    
    print(f"Text: {text}")
    print(f"Response: {list}")
    
    return list, label_counts, response_time, vram_usage, ram_usage_bytes, total_time, response_text

In [53]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'label_counts','response_time', 'vram_usage', 'ram_usage', 'total_time', 'response_text']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_13040\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


Response fields: model, created_at, message, done_reason, done, total_duration, load_duration, prompt_eval_count, prompt_eval_duration, eval_count, eval_duration
{'model': 'deepseek-r1:1.5b', 'created_at': '2025-06-17T02:02:11.0482788Z', 'message': {'role': 'assistant', 'content': 'OUTROS', 'thinking': 'Okay, let me try to figure out how to classify this text based on the given categories: ENTREGA, OUTROS, PRODUTO, CONDICOES DE RECEBIMENTO, ANUNCIO.\n\nFirst, I\'ll read through the text carefully. The main topic seems to be about a product that can still be good but is not recommended by American consumers. There are mentions of issues like problems with shipping and difficulties in receiving it. \n\nThe user also expresses frustration about the product\'s quality and difficulty in getting it delivered. They mention specific problems, such as multiple issues during delivery, which makes them frustrated.\n\nLooking at the categories, "OUTROS" seems to fit because the text is talking abo

In [54]:
for label in labels:
    test[f"{label} pred"] = test.apply(lambda row: 1 if label in row['prediction'] else 0, axis=1)

test = test.drop(columns=['prediction'])

test.to_csv('results/deepseekR1_ZS_multilabel1.csv', index=False)
test

,text,ENTREGA,OUTROS,PRODUTO,CONDICOES DE RECEBIMENTO,ANUNCIO,label_counts,response_time,vram_usage,ram_usage,total_time,response_text,ENTREGA pred,OUTROS pred,PRODUTO pred,CONDICOES DE RECEBIMENTO pred,ANUNCIO pred
0,"ESSE PRODUTO PODE ATÉ SER BOM, PORÉM, A AMERIC...",1,1,0,0,0,"{'ENTREGA': 0, 'OUTROS': 1, 'PRODUTO': 0, 'CON...",5.691375,2313,93.101562,3.657704,"Okay, let me try to figure out how to classify...",0,1,0,0,0
1,Recomendo!Os aparelhos da motorola são muito b...,0,0,1,0,0,"{'ENTREGA': 0, 'OUTROS': 0, 'PRODUTO': 1, 'CON...",6.666891,2313,93.031250,4.607866,"Okay, let me try to figure this out. The user ...",0,0,0,0,0
2,"Eu ameiii o produto, pena que veio com o espe...",0,0,1,1,0,"{'ENTREGA': 0, 'OUTROS': 0, 'PRODUTO': 0, 'CON...",6.219995,2319,92.984375,4.182087,"Okay, let me try to figure this out. The user ...",0,0,0,0,0
3,bom..............................................,0,0,1,0,0,"{'ENTREGA': 0, 'OUTROS': 1, 'PRODUTO': 0, 'CON...",5.229363,2313,93.050781,3.184147,"Okay, let me try to figure this out. The user ...",0,1,0,0,0
4,Quero saber quando chegará mais pois gostaria ...,0,1,0,0,0,{},0.000000,0,0.000000,0.000000,API Error: HTTPConnectionPool(host='localhost'...,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
961,Produto veio com peças totalmente inferiores a...,0,0,0,1,1,"{'ENTREGA': 0, 'OUTROS': 1, 'PRODUTO': 1, 'CON...",5.543229,2480,69.828125,3.485215,"Okay, let me try to figure this out. The user ...",0,1,0,0,0
962,Recebi Produto diferente do anunciado - com ap...,0,0,0,1,1,"{'ENTREGA': 0, 'OUTROS': 0, 'PRODUTO': 0, 'CON...",6.471951,2470,69.832031,4.434541,"Okay, let me try to figure this out. The user ...",0,0,0,0,1
963,No anúncio constava três refis. Foi entregue a...,0,0,0,1,1,"{'ENTREGA': 1, 'OUTROS': 0, 'PRODUTO': 0, 'CON...",6.782431,2470,68.878906,4.741056,"Okay, let me try to figure this out. The user ...",1,0,0,1,0
964,Comprei o produto na cor que está publicado no...,0,0,0,1,1,"{'ENTREGA': 0, 'OUTROS': 0, 'PRODUTO': 1, 'CON...",5.231913,2470,69.386719,3.154828,"Okay, let me try to figure this out. The user ...",0,0,0,0,0


In [55]:
y_true = test[labels].values
y_pred = test[[f"{label} pred" for label in labels]].values

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)
hamming_loss = hamming_loss(y_true, y_pred)
print('Hamming loss: %f' % hamming_loss)

Accuracy: 0.016563
F1 score: 0.189407
Precision: 0.181050
Recall: 0.205111
Hamming loss: 0.394203


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [56]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 6.464776520896896
Average VRAM usage: 2482.3985507246375
Average RAM usage: 74.43956230590062
Average total time: 4.421134146790891


In [57]:
# save results to txt
with open('results/deepseekR1_ZS_multilabel1.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')
    f.write(f'Lines classified: {len(test)}\n')